# Ajax quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YannBerthelot/Ajax/blob/main/notebooks/quickstart.ipynb)

This notebook trains an agent, reads its results, runs the trained policy and saves it. It is the [quickstart](https://yannberthelot.github.io/Ajax/getting-started/quickstart/) of the documentation as a notebook; each cell runs on a CPU in under a minute, compilation included.

In [ ]:
%pip install -q ajax-rl

## Train an agent

An agent takes an environment id and its hyperparameters; `train` runs it. Here PPO learns CartPole with three seeds at once.

In [ ]:
import matplotlib.pyplot as plt

from ajax import PPO
from ajax.logging import LoggingConfig

agent = PPO("CartPole-v1", n_envs=8, n_steps=128)  # 1,024 steps per rollout
# Evaluate every 4,096 steps; backends=() keeps the results in memory only.
logging = LoggingConfig(log_frequency=4_096, backends=())
state, evaluations = agent.train(
    seed=[1, 2, 3], n_timesteps=32_768, logging_config=logging
)

steps = evaluations["timestep"][0]
returns = evaluations["Eval/episodic mean reward"]  # shape (3, 8): seeds x evaluations

plt.plot(steps, returns.mean(axis=0))
plt.fill_between(steps, returns.min(axis=0), returns.max(axis=0), alpha=0.3)
plt.xlabel("environment steps")
plt.ylabel("evaluation return")
plt.show()

`train` compiled the training, environment included, and ran the three seeds side by side. Every 4,096 steps it evaluated the policy on 10 test episodes and stored the mean return: `evaluations` maps every logged name to an array with one row per seed and one column per evaluation.

## Continuous actions

The same pattern works for every agent. SAC on Pendulum, for a few thousand steps: enough to show the mechanics, not to train a pendulum.

In [ ]:
from ajax import SAC

# SAC waits learning_starts steps (10,000 by default) before its first update;
# this short demonstration starts after 1,000.
sac = SAC("Pendulum-v1", learning_starts=1_000)
sac_logging = LoggingConfig(log_frequency=1_000, backends=())
_, sac_evaluations = sac.train(
    seed=[1, 2], n_timesteps=3_000, logging_config=sac_logging
)
print(sac_evaluations["Eval/episodic mean reward"])

## Use the trained policy

`predict` returns the action of one seed's policy for an observation, in the environment's units. It acts as the evaluation acts.

In [ ]:
import gymnax
import jax

env, params = gymnax.make("CartPole-v1")
key = jax.random.PRNGKey(0)
obs, env_state = env.reset(key, params)
episode_return, done = 0.0, False
while not done:
    action = agent.predict(state, obs)  # the first seed's policy
    key, step_key = jax.random.split(key)
    obs, env_state, reward, terminated, truncated, _ = env.step(
        step_key, env_state, action, params
    )
    episode_return += float(reward)
    done = bool(terminated or truncated)
print(episode_return)

## Save and load

`save_checkpoint` writes a state to disk. `load_checkpoint` reads it back into an agent built with the same arguments.

In [ ]:
import os
import tempfile

from ajax.checkpoint import load_checkpoint, save_checkpoint

path = os.path.join(tempfile.mkdtemp(), "ppo_cartpole.pkl")
save_checkpoint(state, path)

agent = PPO("CartPole-v1", n_envs=8, n_steps=128)
state = load_checkpoint(agent, path)
print(agent.predict(state, obs))

## Next

- [Training an agent](https://yannberthelot.github.io/Ajax/guide/training/): seeds, resuming, sweeps.
- [Algorithms](https://yannberthelot.github.io/Ajax/algorithms/): which agent for which problem, and each one's defaults.
- [Logging](https://yannberthelot.github.io/Ajax/guide/logging/): TensorBoard and Weights & Biases.